# Data Ingestion - PostgreSQL
Loads normalized CSVs into PostgreSQL sephora_db.
Creates relational constraints (PRIMARY KEY, FOREIGN KEY).
Tables: products, ingredients, product_ingredients, product_skin_types, reviews

In [1]:
from sqlalchemy import create_engine
import pandas as pd

In [ ]:
engine = create_engine("postgresql://atharva@localhost:5432/sephora_db")

In [3]:
pd.read_sql("SELECT current_user;", engine)

,current_user
0,atharva


In [4]:
products_sql = pd.read_csv("../data/processed/products.csv")
ingredients = pd.read_csv("../data/processed/ingredients.csv")
product_ingredients = pd.read_csv("../data/processed/product_ingredients.csv")
reviews_sql = pd.read_csv("../data/processed/reviews.csv")
product_skin_types = pd.read_csv("../data/processed/product_skin_types.csv")

In [5]:
products_sql.to_sql("products", engine, if_exists="replace", index=False)
ingredients.to_sql("ingredients", engine, if_exists="replace", index=False)
product_ingredients.to_sql("product_ingredients", engine, if_exists="replace", index=False)
reviews_sql.to_sql("reviews", engine, if_exists="replace", index=False)
product_skin_types.to_sql("product_skin_types", engine, if_exists="replace", index=False)

408

In [ ]:
pd.read_sql("select count(*) from ingredients;", engine)
# pd.read_sql("SELECT COUNT(*) FROM product_skin_types;", engine)

,count
0,8156


In [8]:
from sqlalchemy import text

with engine.connect() as conn:
    conn.execute(text("""
        ALTER TABLE products ADD PRIMARY KEY (product_id);
        ALTER TABLE ingredients ADD PRIMARY KEY (ingredient_id);
        ALTER TABLE product_ingredients 
            ADD FOREIGN KEY (product_id) REFERENCES products(product_id),
            ADD FOREIGN KEY (ingredient_id) REFERENCES ingredients(ingredient_id);
        ALTER TABLE product_skin_types 
            ADD FOREIGN KEY (product_id) REFERENCES products(product_id);
    """))
    conn.commit()